In [0]:
### ENTRANCES
# diligenciar UNICAMENTE las 3 variables listadas en la parte de abajo, que corresponden a los snapshots de datos que se van a analizar. Una vez diligenciado, ejecutar este comando con ctrl + Enter, luego, pasamos al siguientes comandos con Alt + tecla ⬇ (flecha hacia abajo) para ejecutar manualmente
snapshot_last = '20241231'
snapshot_prev = '20240930'
snapshot_last_ly = '20231231'

date_snapshot_last = f"{snapshot_last[:4]}-{snapshot_last[4:6]}-{snapshot_last[6:]}" # snapshot_last en formato aaaa-mm-dd
date_snapshot_prev = f"{snapshot_prev[:4]}-{snapshot_prev[4:6]}-{snapshot_prev[6:]}" # snapshot_prev en formato aaaa-mm-dd
date_snapshot_last_ly = f"{snapshot_last_ly[:4]}-{snapshot_last_ly[4:6]}-01" # snapshot_prev en formato aaaa-mm-01

formatted_snapshot_last = f"{snapshot_last[:4]}{snapshot_last[4:6]}" # snapshot_last en formato aaaamm

# print(date_snapshot_last)
# print(formatted_snapshot_last)
# print(date_snapshot_last_ly)


In [0]:
query = f"""
---------------------------------------------------- BRAND LEVEL -----------------------------------------------
with brand_market_lvl as (
  SELECT
    case when brand_country in ('PAN','GTM','BOL','PRY','SLV','CRI','ECU','NIC','HND') then 'CAM' else brand_country end as brand_country,
    brand_code,
    count(distinct brand_customer_id) as Total_CRM_Volume,
    count(distinct case when contactability_key <> 'CONNON' then brand_customer_id end) as Contactable_Contacts,
    0 as New_Contacts_Acquired_in_Quarter,      
    0 as New_Contacts_Acquired_12m,      
    0 as New_Contactable_Contacts_in_Quarter,
    0 as New_Contactable_Contacts_Acquired_12m,
    count(distinct case when lifetime_new_buyer_activity_segment_key not in('DDM_LNBAS_12PB','DDM_LNBAS_NA') then brand_customer_id end) as New_Buyers,
    count(distinct case when lifetime_new_buyer_activity_segment_key not in('DDM_LNBAS_12PB','DDM_LNBAS_NA') and 12_months_unique_orders >= 2 then 
      brand_customer_id end) as New_repeat_Buyers, 
    count(distinct case when lifetime_buyer_activity_segment_key <> 'DDM_LBAS_NA' then brand_customer_id end) as Total_Buyers,  
    count(distinct case when lifetime_buyer_activity_segment_key not in('DDM_LBAS_NA','DDM_LBAS_1324B','DDM_LBAS_2536B','DDM_LBAS_36PB') then 
      brand_customer_id end) as Total_Active_Buyers,  
    (count(distinct case when  lifetime_consumer_activity_segment_key_nonD2C not in('DDM_LCAS_12PC','DDM_LCAS_NA','DDM_LCAS_1324C','DDM_LCAS_36PC', 
      'DDM_LCAS_2536C') then brand_customer_id end) + count(distinct case when lifetime_consumer_activity_segment_key not in ('DDM_LCAS_NA', 'DDM_LCAS_1324C', 'DDM_LCAS_36PC', 'DDM_LCAS_2536C') then brand_customer_id end)) as Total_Engaged_Contacts,
    0 as New_Engaged_Contacts,
    0 as New_Contacts_with_Rich_Data_12m,
    count(distinct case when lifetime_all_enriched_profile = "Y" then brand_customer_id end) as Contacts_With_Rich_data,
    0 as Direct_Messages_Sent_to_New_Contacts_12m,
    0 as Direct_Messages_Sent_to_12m,
    0 as Trigger_Direct_Messages_12m,
    0 as Solicited_contacts_12m,
    0 as New_Solicited_Contacts_12m,
    0 as Engaged_Amongst_Solicited_Contacts_12m,
    0 as DXX_Incremental_Revenue
  from crm_reporting.fact_segment_by_brand
  where  snapshot_date_key = '{snapshot_last}'
  group by all

  UNION ALL

------------------------------------------------------ NEW CONTACTS ------------------------------------------------
  SELECT
    case when brand_country in ('PAN','GTM','BOL','PRY','SLV','CRI','ECU','NIC','HND') then 'CAM' else brand_country end as brand_country,
    brand_code,
    0 as Total_CRM_Volume,
    0 as Contactable_Contacts,
    (count(distinct case when snapshot_date_key = '{snapshot_last}' then brand_customer_id END) 
      - count(distinct case when snapshot_date_key = '{snapshot_prev}' then brand_customer_id END)) AS New_Contacts_Acquired_in_Quarter,
    (count(distinct case when snapshot_date_key = '{snapshot_last}' then brand_customer_id END)
      - count(distinct case when snapshot_date_key = '{snapshot_last_ly}' then brand_customer_id END)) as New_Contacts_Acquired_12m,
    (count(distinct case when snapshot_date_key = '{snapshot_last}' and contactability_key <> 'CONNON'  then brand_customer_id END)
      - count(distinct case when snapshot_date_key = '{snapshot_prev}' and contactability_key <> 'CONNON'  then brand_customer_id END)) AS New_Contactable_Contacts_in_Quarter,
    (count(distinct case when snapshot_date_key = '{snapshot_last}' and contactability_key <> 'CONNON' then brand_customer_id END)
      - count(distinct case when snapshot_date_key = '{snapshot_last_ly}' and contactability_key <> 'CONNON' then brand_customer_id END)) as New_Contactable_Contacts_Acquired_12m,
    0 as New_Buyers,
    0 as New_Repeat_Buyers,
    0 as Total_Buyers,
    0 as Total_Active_Buyers,
    0 as Total_Engaged_Contacts,
    0 as New_Engaged_Contacts,
    0 as New_Contacts_with_Rich_Data_12m,
    0 as Contacts_with_Rich_Data,
    0 as Direct_Messages_Sent_to_New_Contacts_12m,
    0 as Direct_Messages_Sent_to_12m,
    0 as Trigger_Direct_Messages_12m,
    0 as Solicited_contacts_12m,
    0 as New_Solicited_Contacts_12m,
    0 as Engaged_Amongst_Solicited_Contacts_12m,
    0 as DXX_Incremental_Revenue
  FROM prod_latam_catalog.crm_reporting.fact_segment_by_brand
  WHERE snapshot_date_key in ('{snapshot_last_ly}','{snapshot_prev}','{snapshot_last}') 
  GROUP BY ALL

  UNION ALL

------------------------------------------------- DIRECT MESSAGES & SOLICITED ---------------------------------------------
  SELECT
    case when brand_country in ('PAN','GTM','BOL','PRY','SLV','CRI','ECU','NIC','HND') then 'CAM' else brand_country end as brand_country,
    brand_code,
    0 as Total_CRM_Volume,
    0 as Contactable_Contacts,
    0 as New_Contacts_Acquired_in_Quarter,
    0 as New_Contacts_Acquired_12m,
    0 as New_Contactable_Contacts_in_Quarter,
    0 as New_Contactable_Contacts_Acquired_12m,
    0 as New_Buyers,
    0 as New_Repeat_Buyers,
    0 as Total_Buyers,
    0 as Total_Active_Buyers,
    0 as Total_Engaged_Contacts,
    0 as New_Engaged_Contacts,
    0 as New_Contacts_with_Rich_Data_12m,
    0 as Contacts_with_Rich_Data,
    0 as Direct_Messages_Sent_to_New_Contacts_12m,
    sum(sent_cnt) as Direct_Messages_Sent_to_12m,
    0 as Trigger_Direct_Messages_12m,
    count(distinct case when sent_cnt = 1 and bounced_cnt = 0 then brand_customer_id END) as Solicited_contacts_12m,
    0 as New_Solicited_Contacts_12m,
    count(distinct case when sent_cnt = 1 and (unique_click_cnt = 1 or unique_open_cnt =1) then brand_customer_id END) as 
      Engaged_Amongst_Solicited_Contacts_12m,
    0 as DXX_Incremental_Revenue
  FROM prod_latam_catalog.crm_reporting.fact_campaign_interaction
  WHERE msg_channel ='EMAIL'
    and brand_customer_id is not null
    and (CAST(CONCAT(sent_year, '-', sent_month, '-01') AS DATE) 
      BETWEEN '{date_snapshot_last_ly}' AND '{date_snapshot_last}')
  group by all

  
  UNION ALL

------------------------------------------------- TRIGGER ---------------------------------------------
  SELECT
    case when brand_country in ('PAN','GTM','BOL','PRY','SLV','CRI','ECU','NIC','HND') then 'CAM' else brand_country end as brand_country,
    brand_code,
    0 as Total_CRM_Volume,
    0 as Contactable_Contacts,
    0 as New_Contacts_Acquired_in_Quarter,
    0 as New_Contacts_Acquired_12m,
    0 as New_Contactable_Contacts_in_Quarter,
    0 as New_Contactable_Contacts_Acquired_12m,
    0 as New_Buyers,
    0 as New_Repeat_Buyers,
    0 as Total_Buyers,
    0 as Total_Active_Buyers,
    0 as Total_Engaged_Contacts,
    0 as New_Engaged_Contacts,
    0 as New_Contacts_with_Rich_Data_12m,
    0 as Contacts_with_Rich_Data,
    0 as Direct_Messages_Sent_to_New_Contacts_12m,
    0 as Direct_Messages_Sent_to_12m,
    SUM(sent_cnt) as Trigger_Direct_Messages_12m,
    0 as Solicited_contacts_12m,
    0 as New_Solicited_Contacts_12m,
    0 as Engaged_Amongst_Solicited_Contacts_12m,
    0 as DXX_Incremental_Revenue
  FROM prod_latam_catalog.crm_analytics.fact_journey_campaign
  WHERE (CAST(CONCAT(year, '-', month, '-01') AS DATE) 
      BETWEEN '{date_snapshot_last_ly}' AND '{date_snapshot_last}')
  group by all
)


select 
  a.brand_country,
  case 
    when a.brand_country = 'ARG' then 'Argentina'
    when a.brand_country = 'BRA' then 'Brazil'
    when a.brand_country = 'CHI' then 'Chile'
    when a.brand_country = 'CAM' then 'Central America'
    when a.brand_country = 'PER' then 'Peru'
    when a.brand_country = 'COL' then 'Colombia'
    when a.brand_country = 'URU' then 'Uruguay'
    when a.brand_country = 'MEX' then 'Mexico' end as Market,
  a.brand_code,
  b.brand_name as Brand,
  case when b.d2c_flag = 'Y' then 'Yes' else 'No' end as D2C_Flag,
  sum(Total_CRM_Volume) as Total_CRM_Volume,
  sum(Contactable_Contacts) as Contactable_Contacts,
  sum(New_Contacts_Acquired_in_Quarter) as New_Contacts_Acquired_in_Quarter,
  sum(New_Contacts_Acquired_12m) as New_Contacts_Acquired_12m,
  ---- AJUSTE MANUAL ----
  case 
    when sum(New_Contactable_Contacts_in_Quarter) > sum(New_Contacts_Acquired_in_Quarter) then sum(New_Contacts_Acquired_in_Quarter)
    when sum(New_Contactable_Contacts_in_Quarter) < 0 then 0 
    else sum(New_Contactable_Contacts_in_Quarter) end as New_Contactable_Contacts_in_Quarter,
  case 
    when sum(New_Contactable_Contacts_Acquired_12m) > sum(New_Contacts_Acquired_12m) then sum(New_Contacts_Acquired_12m)
    when sum(New_Contactable_Contacts_Acquired_12m) < 0 then 0 
    else sum(New_Contactable_Contacts_Acquired_12m) end as New_Contactable_Contacts_Acquired_12m,
  -------------------------------
  sum(New_Buyers) as New_Buyers,
  sum(New_Repeat_Buyers) as New_Repeat_Buyers,
  sum(Total_Buyers) as Total_Buyers,
  sum(Total_Active_Buyers) as Total_Active_Buyers,
  sum(Total_Engaged_Contacts) as Total_Engaged_Contacts,
  sum(New_Engaged_Contacts) as New_Engaged_Contacts,
  sum(New_Contacts_with_Rich_Data_12m) as New_Contacts_with_Rich_Data_12m,
  sum(Contacts_with_Rich_Data) as Contacts_with_Rich_Data,
  sum(Direct_Messages_Sent_to_New_Contacts_12m) as Direct_Messages_Sent_to_New_Contacts_12m,
  sum(Direct_Messages_Sent_to_12m) as Direct_Messages_Sent_to_12m,
  sum(Trigger_Direct_Messages_12m) as Trigger_Direct_Messages_12m,
  sum(Solicited_contacts_12m) as Solicited_contacts_12m,
  sum(New_Solicited_Contacts_12m) as New_Solicited_Contacts_12m,
  sum(Engaged_Amongst_Solicited_Contacts_12m) as Engaged_Amongst_Solicited_Contacts_12m,
  sum(DXX_Incremental_Revenue) as DXX_Incremental_Revenue
from brand_market_lvl a
left join prod_latam_catalog.crm_analytics.dim_brand_workbook b
  on a.brand_country = b.brand_country
  and a.brand_code = b.brand_code
group by all
order by Market,Brand
"""

brand_lvl = spark.sql(query)
brand_lvl.createOrReplaceTempView("brand_lvl_vw")

brand_lvl = brand_lvl.drop('brand_country','brand_code')
display(brand_lvl)

Market,Brand,D2C_Flag,Total_CRM_Volume,Contactable_Contacts,New_Contacts_Acquired_in_Quarter,New_Contacts_Acquired_12m,New_Contactable_Contacts_in_Quarter,New_Contactable_Contacts_Acquired_12m,New_Buyers,New_Repeat_Buyers,Total_Buyers,Total_Active_Buyers,Total_Engaged_Contacts,New_Engaged_Contacts,New_Contacts_with_Rich_Data_12m,Contacts_with_Rich_Data,Direct_Messages_Sent_to_New_Contacts_12m,Direct_Messages_Sent_to_12m,Trigger_Direct_Messages_12m,Solicited_contacts_12m,New_Solicited_Contacts_12m,Engaged_Amongst_Solicited_Contacts_12m,DXX_Incremental_Revenue
Argentina,Armani,No,73356,68067,935,15313,630,14467,0,0,0,0,15879,0,0,3695,0,0,0,0,0,0,0
Argentina,Biotherm,No,10104,9967,0,695,0,695,0,0,0,0,723,0,0,20,0,0,0,0,0,0,0
Argentina,CeraVe,No,104351,94703,914,24914,35,22010,0,0,0,0,30661,0,0,61411,0,1286181,0,85764,0,23664,0
Argentina,Dermacenter,No,375,368,0,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0
Argentina,Kiehl's,Yes,101362,69642,2694,16223,690,8017,5528,894,23060,8490,60090,0,0,19758,0,8114109,98049,68742,0,45326,0
Argentina,Kérastase,No,436153,388308,13183,210156,9885,200975,0,0,0,0,223954,0,0,293598,0,2767900,0,376947,0,67723,0
Argentina,L'Oréal Pro,No,418163,356767,26140,172052,21446,161815,0,0,0,0,186724,0,0,289942,0,3700633,0,349285,0,78995,0
Argentina,La Roche Posay,No,596435,505078,27811,160949,24005,142678,0,0,0,0,214736,0,0,393778,0,18197294,0,471247,0,178203,0
Argentina,Lancome,Yes,385632,294126,7298,43511,1435,26397,7450,757,40913,10151,210241,0,0,49100,0,23857378,339133,275880,0,153457,0
Argentina,Matrix,No,24793,23210,3,11,0,0,0,0,0,0,245,0,0,19535,0,77651,0,21171,0,1579,0


In [0]:
# %sql
# describe crm_reporting.fact_segment_by_brand

In [0]:
query = f"""
---------------------------------------------------- MARKET LEVEL -----------------------------------------------
with multibrand_1 as (
  select 
    case when brand_country in ('PAN','GTM','BOL','PRY','SLV','CRI','ECU','NIC','HND') then 'CAM' else brand_country end as brand_country,
    global_customer_id, 
    count(distinct brand_customer_id) as counts
  from prod_latam_catalog.crm_reporting.fact_segment_by_brand
  WHERE snapshot_date_key in ('{snapshot_last}') 
  group by all
  having counts > 1
),

unions as (
  select 
    case 
      when brand_country = 'ARG' then 'Argentina'
      when brand_country = 'BRA' then 'Brazil'
      when brand_country = 'CHI' then 'Chile'
      when brand_country = 'CAM' then 'Central America'
      when brand_country = 'PER' then 'Peru'
      when brand_country = 'COL' then 'Colombia'
      when brand_country = 'URU' then 'Uruguay'
      when brand_country = 'MEX' then 'Mexico' end as Market,
    0 as Total_CRM_Contacts,
    0 as Contactable_Contacts,
    0 as New_Contacts_Acquired_in_Quarter,
    0 as New_Contacts_Acquired_12m,
    0 as New_Contactable_Contacts_in_Quarter,
    0 as New_Contactable_Contacts_Acquired_12m,
    0 as New_Buyers,
    0 as New_Repeat_Buyers,
    0 as Total_Buyers,
    0 as Total_Active_Buyers,
    0 as Total_Engaged_Contacts,
    count(global_customer_id) as Multi_Brand_Contacts
  from multibrand_1
  group by all

  union all

  select 
    Market,
    sum(Total_CRM_Volume) as Total_CRM_Contacts,
    sum(Contactable_Contacts) as Contactable_Contacts,
    sum(New_Contacts_Acquired_in_Quarter) as New_Contacts_Acquired_in_Quarter,
    sum(New_Contacts_Acquired_12m) as New_Contacts_Acquired_12m,
    sum(New_Contactable_Contacts_in_Quarter) as New_Contactable_Contacts_in_Quarter,
    sum(New_Contactable_Contacts_Acquired_12m) as New_Contactable_Contacts_Acquired_12m,
    sum(New_Buyers) as New_Buyers,
    sum(New_Repeat_Buyers) as New_Repeat_Buyers,
    sum(Total_Buyers) as Total_Buyers,
    sum(Total_Active_Buyers) as Total_Active_Buyers,
    sum(Total_Engaged_Contacts) as Total_Engaged_Contacts,
    0 as Multi_Brand_Contacts
  from brand_lvl_vw
  group by all
  order by Market
)

select 
  Market,
  'No' as Unique_numbers,
  sum(Total_CRM_Contacts) as Total_CRM_Contacts,
  sum(Contactable_Contacts) as Contactable_Contacts,
  sum(New_Contacts_Acquired_in_Quarter) as New_Contacts_Acquired_in_Quarter,
  sum(New_Contacts_Acquired_12m) as New_Contacts_Acquired_12m,
  sum(New_Contactable_Contacts_in_Quarter) as New_Contactable_Contacts_in_Quarter,
  sum(New_Contactable_Contacts_Acquired_12m) as New_Contactable_Contacts_Acquired_12m,
  sum(New_Buyers) as New_Buyers,
  sum(New_Repeat_Buyers) as New_Repeat_Buyers,
  sum(Total_Buyers) as Total_Buyers,
  sum(Total_Active_Buyers) as Total_Active_Buyers,
  sum(Total_Engaged_Contacts) as Total_Engaged_Contacts,
  sum(Multi_Brand_Contacts) as Multi_Brand_Contacts
from unions
group by all
order by Market


"""

market_lvl = spark.sql(query)
display(market_lvl)
# market_lvl.createOrReplaceTempView("market_lvl_vw")

Market,Total_CRM_Contacts,Contactable_Contacts,New_Contacts_Acquired_in_Quarter,New_Contacts_Acquired_12m,New_Contactable_Contacts_in_Quarter,New_Contactable_Contacts_Acquired_12m,New_Buyers,New_Repeat_Buyers,Total_Buyers,Total_Active_Buyers,Total_Engaged_Contacts,Multi_Brand_Contacts
Argentina,2572511,2148786,80595,667593,58480,591038,12978,1651,63973,18641,1009869,338516
Brazil,9058757,5463376,388716,1447841,254228,915048,639594,188672,3420810,1682557,4612114,516683
Central America,138225,135898,6918,63582,6832,62885,0,0,0,0,68523,5700
Chile,2678779,2301815,110163,640410,103730,581870,46072,5827,328370,70070,1236479,402843
Colombia,418779,396130,53282,212208,52898,206611,0,0,0,0,223058,34065
Mexico,9296937,7501452,749805,2658571,717972,2469229,49322,6551,353002,80201,3420006,1516969
Peru,439656,394034,32775,130401,30777,121358,0,0,0,0,149182,47246
Uruguay,581458,531505,30693,195526,28452,194573,9982,1932,25137,14023,281163,98190


In [0]:
## SERVICIOS
### Merged tables
services = spark.sql("""
select 
  -- brand_country,
  case 
    when a.brand_country = 'ARG' then 'Argentina'
    when a.brand_country = 'BRA' then 'Brazil'
    when a.brand_country = 'CHI' then 'Chile'
    when a.brand_country = 'CAM' then 'Central America'
    when a.brand_country = 'PER' then 'Peru'
    when a.brand_country = 'COL' then 'Colombia'
    when a.brand_country = 'URU' then 'Uruguay'
    when a.brand_country = 'MEX' then 'Mexico' end as Market,
  -- brand_code,
  b.brand_name as Brand,
  case when b.d2c_flag = 'Y' then 'Yes' else 'No' end as D2C_Flag,
  upper(source_name) as source_name,
  lower(registration_sub_source) AS registration_sub_source,
  count(distinct source_customer_id) as ids,
  max(date(created_dt)) as created_dt
from prod_latam_catalog.crm_reporting.dim_customer a
left join prod_latam_catalog.crm_analytics.dim_brand_workbook b
  on a.brand_country = b.brand_country
  and a.brand_code = b.brand_code
where upper(source_name) IN ('DEMANDWARE','SITECORE') 
  AND (lower(registration_source) LIKE ('%quiz%') 
     OR lower(registration_source) LIKE ('%diagnos%')
     OR lower(registration_source) LIKE ('%website%'))
  AND lower(registration_sub_source) NOT IN ('cart checkout','header','','сart checkout','footer','popup')
  -- AND to_date(a.created_dt) between '2024-10-01' and '2024-12-31'
group by all
order by Market,Brand,registration_sub_source

""")

display(services)

# print(merged_gdm.count()) #303918

Market,Brand,D2C_Flag,source_name,registration_sub_source,ids,created_dt
Argentina,Kérastase,No,SITECORE,hair quiz,9350,2025-03-12
Brazil,Dermaclub,Yes,DEMANDWARE,skin dr,2493,2024-09-05
Brazil,Kérastase,Yes,DEMANDWARE,hair quiz,36074,2025-03-12
Brazil,La Roche Posay,No,DEMANDWARE,product finder,60072,2024-07-15
Brazil,La Roche Posay,No,DEMANDWARE,skin dr,1890,2025-03-12
Brazil,Vichy,No,DEMANDWARE,skin dr,5,2024-08-08
Chile,Kiehl's,Yes,DEMANDWARE,skin dr,4,2024-01-23
Chile,Kérastase,Yes,DEMANDWARE,hair quiz,17537,2025-03-12
Chile,La Roche Posay,No,DEMANDWARE,skin dr,2,2025-03-12
Chile,Lancome,Yes,DEMANDWARE,product finder,23,2024-09-30


In [0]:
# -------------------------------------------------   VALIDACIONES  ------------------------------------------------------------------

In [0]:
%sql
select 
  brand_country,
  brand_code,
  upper(source_name) as source_name,
  registration_source,
  lower(registration_sub_source) AS registration_sub_source,
  count(distinct source_customer_id) as ids,
  max(date(created_dt)) as created_dt
from prod_latam_catalog.crm_reporting.dim_customer
where upper(source_name) IN ('DEMANDWARE','SITECORE')
  and brand_country = 'MEX'
  and brand_code = 'SKI'
group by all

brand_country,brand_code,source_name,registration_source,registration_sub_source,ids,created_dt
MEX,SKI,DEMANDWARE,Newsletter,registration,7,2023-02-09
MEX,SKI,DEMANDWARE,null,registration,130,2022-12-04
MEX,SKI,DEMANDWARE,Newsletter,сart checkout,1908,2025-03-09
MEX,SKI,DEMANDWARE,CreateOrder,null,3813,2022-02-16
MEX,SKI,DEMANDWARE,Newsletter,null,3,2024-10-31
MEX,SKI,DEMANDWARE,Footer,null,4238,2022-02-16
MEX,SKI,DEMANDWARE,Registration,null,11232,2025-03-10
MEX,SKI,DEMANDWARE,null,null,32942,2025-03-08
MEX,SKI,DEMANDWARE,Newsletter,header,68,2025-02-23
MEX,SKI,DEMANDWARE,Modal,null,12284,2022-02-16


In [0]:
%sql
select brand_country,brand_code,event_dt, campaign_id,campaign_name
from crm_reporting.fact_message_tracker
where source_customer_id  in ('0c5e478bd2e5920b103fa03ebbd6d9d2',
'a1b79d1421ed7dfa97ab4cb4b59ae775')

brand_country,brand_code,event_dt,campaign_id,campaign_name
BRA,LAN,2024-11-13T05:02:51Z,AOZ6,AOZ6_LAN_BRA_LN-MC-53-WHLA13-N-313-SEG_1_2024
BRA,LAN,2024-11-15T05:01:50Z,APE4,APE4_LAN_BRA_LN-MQ-00-CLGI15-N-306-SEG_1_2024
BRA,LAN,2024-11-18T05:01:42Z,APC3,APC3_LAN_BRA_LN-MC-02-TYOF18-E-306-SEG_1_2024
BRA,LAN,2024-11-20T17:31:50Z,APF2,APF2_LAN_BRA_LN-MQ-02-FSLA20-N-002-SEG_1_2024
BRA,LAN,2024-11-22T15:01:43Z,APH3,APH3_LAN_BRA_LN-MQ-00-MHLN22-N-306-SEG_1_2024
BRA,LAN,2024-11-17T05:01:46Z,APB9,APB9_LAN_BRA_LN-MC-02-OWLA17-N-002-SEG_1_2024
BRA,LAN,2024-11-24T05:01:49Z,APG3,APG3_LAN_BRA_LN-MC-02-OSLA24-N-306-SEG_1_2024
BRA,LAN,2024-11-25T05:01:42Z,API5,API5_LAN_BRA_LN-MC-02-LCLA25-N-313-SEG_1_2024
BRA,LAN,2024-11-25T05:01:42Z,API5,API5_LAN_BRA_LN-MC-02-LCLA25-N-313-SEG_1_2024
BRA,LAN,2024-11-27T05:01:55Z,APJ8,APJ8_LAN_BRA_LN-MC-00-BETC27-E-313-SEG_1_2024
